# 🎵 Music Streaming Analysis: Springfield vs. Shelbyville

**Project 2 — Exploratory Data Analysis**

# Table of Contents <a id='back'></a>

* [Introduction](#intro)
* [Step 1. Data Overview](#data_review)
    * [Conclusions](#data_review_conclusions)
* [Step 2. Data Preprocessing](#data_preprocessing)
    * [2.1 Column Header Formatting](#header_style)
    * [2.2 Missing Values](#missing_values)
    * [2.3 Duplicates](#duplicates)
    * [2.4 Conclusions](#data_preprocessing_conclusions)
* [Step 3. Hypothesis Testing](#hypothesis)
    * [3.1 Hypothesis: User Activity in Both Cities](#activity)
* [Final Conclusions](#end)

## Introduction <a id='intro'></a>

The role of a data analyst is to extract valuable insights from data and support evidence-based decision-making. This process involves several stages: data overview, data preprocessing, and hypothesis testing.

Every research project begins with a hypothesis that can later be tested — and either accepted or rejected. For a business to make sound decisions, it must be able to evaluate whether its assumptions hold up against real data.

In this project, we compare the music preferences of users in **Springfield** and **Shelbyville** using data from an online music streaming service. The goal is to test the hypothesis below and understand user behavior across the two cities.

### Objective

Test the following hypothesis:
> **User activity differs depending on the day of the week and the city.**

### Steps
The data is stored in `/datasets/music_project_en.csv`. Since the quality of the data is unknown, we will examine it before testing the hypothesis.

The project is divided into three stages:
1. Data Overview
2. Data Preprocessing
3. Hypothesis Testing

[Back to Table of Contents](#back)

## Step 1. Data Overview <a id='data_review'></a>

Let's open the dataset and explore its structure.

Import `pandas`, which will be used throughout the analysis:

In [ ]:
import pandas as pd  # importing pandas

Read the dataset from `/datasets/music_project_en.csv` and store it in the variable `df`:

In [ ]:
df = pd.read_csv('/datasets/music_project_en.csv')

Display the first 10 rows to get an initial look at the data:

In [ ]:
df.head(10)  # display the first 10 rows

Get general information about the dataset using `.info()`:

In [ ]:
df.info()  # general info about the dataset

**Observations:**

The table has seven columns, all storing `object` (string) type data.

According to the documentation:
- `' userID'` — user identifier
- `'Track'` — track title
- `'artist'` — artist name
- `'genre'` — music genre
- `'City'` — user's city
- `'time'` — exact time the track was played
- `'Day'` — day of the week

We can identify three style issues in the column headers:
1. Inconsistent capitalization (some headers are uppercase, others are lowercase).
2. Some headers contain extra whitespace.
3. The column `userID` uses camelCase instead of `snake_case`.

### Observations <a id='data_review_conclusions'></a>

**1. What type of data do we have, and what do the columns represent?**

All columns store `object` (string) data:
- `' userID'` — user identifier
- `'Track'` — track title
- `'artist'` — artist name
- `'genre'` — music genre
- `'City'` — user's city
- `'time'` — exact playback time
- `'Day'` — day of the week

**2. Is this data sufficient to test our hypothesis?**

Yes. We have city, day, and user activity data — enough to compare user behavior across cities and days.

**3. Are there any visible data quality issues?**

Yes:
- Inconsistent capitalization across column names (`Track`, `artist`, `City`, `Day`).
- Extra whitespace in some column names (e.g., `' userID'`, `' City'`).
- Column naming is not standardized (camelCase vs. lowercase).

[Back to Table of Contents](#back)

## Step 2. Data Preprocessing <a id='data_preprocessing'></a>

The goal here is to prepare the data for analysis. We'll start by fixing the column headers, then handle missing values and duplicates.

### 2.1 Column Header Formatting <a id='header_style'></a>

Let's print the column headers to inspect them:

In [ ]:
df.columns  # print column names

We'll update the headers following good naming practices:
- All characters in **lowercase**
- No leading or trailing **whitespace**
- Use **snake_case** for multi-word names

**Step 1:** Convert all column names to lowercase using a `for` loop:

In [ ]:
new_columns = []

for col in df.columns:
    new_columns.append(col.lower())

df.columns = new_columns
df.columns  # verify lowercase conversion

**Step 2:** Strip leading and trailing whitespace from each column name:

In [ ]:
new_columns = []

for col in df.columns:
    new_columns.append(col.strip())

df.columns = new_columns
df.columns  # verify whitespace removal

**Step 3:** Rename `userid` to `user_id` to follow snake_case convention:

In [ ]:
df = df.rename(columns={'userid': 'user_id'})
df.columns  # verify final column names

[Back to Table of Contents](#back)

### 2.2 Missing Values <a id='missing_values'></a>

First, let's count missing values in each relevant column:

In [ ]:
for col in ['track', 'artist', 'genre']:
    print(f'Missing values in {col}:', df[col].isna().sum())

Not all missing values are equally critical:
- Missing values in `track` and `artist` are not critical — we can safely replace them with a default value.
- Missing values in `genre` **could affect** the comparison of music preferences between cities. Ideally, we would investigate why they are missing, but for this project we will fill them with a default value.

We'll replace missing values in `track`, `artist`, and `genre` with the string `'unknown'`:

In [ ]:
# Check missing values before filling
print("Missing values before filling:")
for col in ['track', 'artist', 'genre']:
    print(f'  {col}:', df[col].isna().sum())

# Replace missing values with 'unknown'
for col in ['track', 'artist', 'genre']:
    df[col] = df[col].fillna('unknown')

# Verify no missing values remain
print("\nMissing values after filling:")
for col in ['track', 'artist', 'genre']:
    print(f'  {col}:', df[col].isna().sum())

Now let's confirm the entire dataset is free of missing values:

In [ ]:
df.isna().sum()  # check missing values across all columns

[Back to Table of Contents](#back)

### 2.3 Duplicates <a id='duplicates'></a>

Let's count the number of explicit duplicate rows in the dataset:

In [ ]:
df.duplicated().sum()  # count explicit duplicates

Remove all duplicate rows:

In [ ]:
df = df.drop_duplicates()  # drop explicit duplicates

Verify that no duplicates remain:

In [ ]:
df.duplicated().sum()  # confirm duplicates were removed

Now let's look for **implicit duplicates** in the `genre` column. Genre names may appear with different spellings or abbreviations. Let's print the sorted list of unique genres:

In [ ]:
df['genre'].unique()  # list all unique genre values

By inspecting the list, we can identify the following implicit duplicates for `hiphop`:
- `hip`
- `hop`
- `hip-hop`

We'll create a function `replace_wrong_genres()` to fix these:

In [ ]:
def replace_wrong_genres(wrong_genres, correct_genre):
    """Replace incorrect genre names with the correct standardized name."""
    for genre in wrong_genres:
        df['genre'] = df['genre'].replace(genre, correct_genre)  # replace implicit duplicates

Call the function to fix all `hiphop` variants:

In [ ]:
replace_wrong_genres(['hip', 'hop', 'hip-hop'], 'hiphop')  # fix implicit duplicates

Verify the result by printing the sorted unique genre list again:

In [ ]:
print(sorted(df['genre'].unique()))  # confirm duplicates were resolved

[Back to Table of Contents](#back)

### 2.4 Conclusions <a id='data_preprocessing_conclusions'></a>

During preprocessing, we identified and resolved the following issues:

- **Column headers:** Standardized to lowercase with no whitespace. Renamed `userid` → `user_id`.
- **Missing values:** Found in `track`, `artist`, and `genre` columns. Replaced with `'unknown'` using a loop and `.fillna()`.
- **Explicit duplicates:** Detected and removed using `.drop_duplicates()`.
- **Implicit duplicates:** Found three genre name variants (`hip`, `hop`, `hip-hop`) representing the same genre. Replaced with the canonical value `hiphop` using a custom function.

[Back to Table of Contents](#back)

## Step 3. Hypothesis Testing <a id='hypothesis'></a>

### Hypothesis: User Activity Comparison Between Cities <a id='activity'></a>

**Hypothesis:** There are differences in music consumption between users in Springfield and Shelbyville.

To test this, we'll analyze listening activity on three days of the week: **Monday**, **Wednesday**, and **Friday**. The approach:
1. Group users by city.
2. Compare the number of tracks played in each city on each of the three days.

**Step 1:** Evaluate total user activity per city:

In [ ]:
df.groupby(by='city')['user_id'].count()  # count tracks played per city

**Observation:** Springfield has a considerably higher total number of plays compared to Shelbyville.

**Step 2:** Group by day of the week and compare activity across Monday, Wednesday, and Friday:

In [ ]:
df.groupby(by='day')['user_id'].count()  # count tracks played per day

**Observation:** Friday shows the highest listening activity among the three days.

**Step 3:** Create a function `number_tracks()` to count plays filtered by both **day** and **city** simultaneously:

In [ ]:
def number_tracks(day, city):
    """Return the number of tracks played on a given day in a given city."""
    day_filtered = df[df['day'] == day]
    city_filtered = day_filtered[day_filtered['city'] == city]
    track_count = city_filtered['user_id'].count()
    return track_count

Now call the function six times to compare both cities across all three days:

In [ ]:
number_tracks('Monday', 'Springfield')  # tracks played in Springfield on Monday

In [ ]:
number_tracks('Monday', 'Shelbyville')  # tracks played in Shelbyville on Monday

In [ ]:
number_tracks('Wednesday', 'Springfield')  # tracks played in Springfield on Wednesday

In [ ]:
number_tracks('Wednesday', 'Shelbyville')  # tracks played in Shelbyville on Wednesday

In [ ]:
number_tracks('Friday', 'Springfield')  # tracks played in Springfield on Friday

In [ ]:
number_tracks('Friday', 'Shelbyville')  # tracks played in Shelbyville on Friday

**Conclusions**

The results confirm that Springfield consistently shows higher listening activity than Shelbyville across all three days. Both cities exhibit peak activity on Fridays, with a noticeable drop mid-week (Wednesday).

[Back to Table of Contents](#back)

# Final Conclusions <a id='end'></a>

Springfield shows a considerably higher volume of user activity compared to Shelbyville across all days analyzed. Based on this, **the hypothesis is confirmed**: user activity differs depending on both the day of the week and the city.

**Key findings:**
- Springfield consistently outperforms Shelbyville in total track plays.
- Friday is the most active day in both cities.
- The gap between cities is visible across all three days (Monday, Wednesday, Friday).

> **Note:** In real-world research projects, hypothesis testing requires more rigorous statistical methods (e.g., t-tests, chi-square tests). Additionally, conclusions about an entire city's listening behavior cannot always be drawn from a single data source. These findings are exploratory in nature.

You will learn more about statistical hypothesis testing in the data analysis sprint.

[Back to Table of Contents](#back)